In [10]:
import os, glob

DATA_DIR = "project_data_mm"   
PDF_DIR  = os.path.join(DATA_DIR, "pdfs", )
FIG_DIR  = os.path.join(DATA_DIR, "images")

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

TOP_K_TEXT     = 5    
TOP_K_IMAGES   = 3    
TOP_K_EVIDENCE = 8    
ALPHA = 0.5  

pdfs = sorted(
    glob.glob(os.path.join(PDF_DIR, "*.pdf")) +
    glob.glob(os.path.join(PDF_DIR, "*.PDF"))
)
imgs = sorted(glob.glob(os.path.join(FIG_DIR, "*.*")))
print("PDFs:", len(pdfs))
print("Images:", len(imgs))

PDFs: 5
Images: 0


In [11]:
import os
import re
import json
import time
from typing import List, Dict, Tuple, Optional, Any

import pandas as pd
import numpy as np

import fitz  # PyMuPDF
import pytesseract
from pdf2image import convert_from_path
from PIL import Image

from sentence_transformers import SentenceTransformer, CrossEncoder

import faiss

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer

from dotenv import load_dotenv
from groq import Groq

load_dotenv()

/Users/ahirmansi1301gmail.com/LexGaurd/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [12]:
QUERIES = [
    {
        "id": "Q1",
        "question": "Under the Co-Branding Agreement, what is the exact formula used to calculate the advertising fees i-Escrow must pay 2TheMart?",
        "rubric": {
            "must_have_keywords": ["0.025%", "average transaction"],
            "optional_keywords": ["gross revenue", "monthly", "advertising fee", "co-branding"]
        }
    },
    {
        "id": "Q2",
        "question": "What are the specific conditions and timeline under which Adams Golf can terminate the endorsement agreement with Tom Watson for cause?",
        "rubric": {
            "must_have_keywords": ["material breach", "15 days", "written notice"],
            "optional_keywords": ["cure", "termination", "cause", "default"]
        }
    },
    {
        "id": "Q3",
        "question": "According to the Adams Golf Endorsement Agreement, what specific brand of putter is Tom Watson contractually required to use during the US Open?",
        "rubric": {
            "must_have_keywords": ["not required", "other manufacturer", "exception"],
            "optional_keywords": ["putter", "US Open", "equipment", "exclusivity"]
        }
    },
]

In [18]:
from dataclasses import dataclass
@dataclass
class TextChunk:
    chunk_id: str
    doc_id: str
    page_num: int
    text: str

@dataclass
class ImageItem:
    item_id: str
    path: str
    caption: str

def clean_text(s: str) -> str:
    s = s or ""
    s = re.sub(r"\s+", " ", s).strip()
    return s

def extract_pdf_pages(pdf_path: str) -> List[TextChunk]:
    doc_id = os.path.basename(pdf_path)
    doc = fitz.open(pdf_path)
    out: List[TextChunk] = []
    for i in range(len(doc)):
        page = doc.load_page(i)
        text = clean_text(page.get_text("text"))
        if len(text) < 50:
            try:
                images = convert_from_path(pdf_path, first_page=i+1, last_page=i+1, dpi=300)
                if images:
                    ocr_text = pytesseract.image_to_string(images[0])
                    text = clean_text(ocr_text)
            except Exception as e:
                print(f"OCR fallback failed for {doc_id} page {i+1}: {e}")
        if text:
            out.append(TextChunk(
                chunk_id=f"{doc_id}::p{i+1}",
                doc_id=doc_id,
                page_num=i+1,
                text=text
            ))
    return out

def load_images(fig_dir: str) -> List[ImageItem]:
    items: List[ImageItem] = []
    if not os.path.isdir(fig_dir):
        return items
    for p in sorted(glob.glob(os.path.join(fig_dir, "*.*"))):
        ext = os.path.splitext(p)[1].lower()
        if ext in {".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".gif"}:
            base = os.path.basename(p)
            caption = os.path.splitext(base)[0].replace("_", " ")
            items.append(ImageItem(item_id=base, path=p, caption=caption))
    return items
    # Run the ingestion process
page_chunks = []
for p in pdfs:
    page_chunks.extend(extract_pdf_pages(p))

image_items = load_images(FIG_DIR)

print(f"Total text chunks extracted: {len(page_chunks)}")
print(f"Total images loaded: {len(image_items)}")


Total text chunks extracted: 64
Total images loaded: 0


In [19]:
# --- Text Retrieval Engine ---

text_model = SentenceTransformer('all-MiniLM-L6-v2')

# BM25 index
tokenized_corpus = [chunk.text.lower().split() for chunk in page_chunks]
bm25_index = BM25Okapi(tokenized_corpus)

# FAISS index
chunk_embeddings = text_model.encode(
    [chunk.text for chunk in page_chunks],
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

faiss.normalize_L2(chunk_embeddings)
faiss_dim = chunk_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(faiss_dim)
faiss_index.add(chunk_embeddings)

def _minmax(arr: np.ndarray) -> np.ndarray:
    mn, mx = arr.min(), arr.max()
    if mx - mn < 1e-9:
        return np.zeros_like(arr)
    return (arr - mn) / (mx - mn)

def hybrid_retrieve(query: str, top_k: int = 5, alpha: float = 0.5) -> List[Tuple[int, float]]:
    # BM25 scores
    bm25_scores = np.array(bm25_index.get_scores(query.lower().split()), dtype="float32")

    # FAISS scores
    q_emb = text_model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_emb)
    raw_faiss_scores = np.zeros(len(page_chunks), dtype="float32")
    distances, indices = faiss_index.search(q_emb, len(page_chunks))
    for rank, idx in enumerate(indices[0]):
        raw_faiss_scores[idx] = distances[0][rank]

    # Normalize and fuse
    norm_bm25 = _minmax(bm25_scores)
    norm_faiss = _minmax(raw_faiss_scores)
    fused = alpha * norm_bm25 + (1 - alpha) * norm_faiss

    top_indices = np.argsort(fused)[::-1][:top_k]
    return [(int(i), float(fused[i])) for i in top_indices]

# --- Image Retrieval Engine (TF-IDF on captions) ---

captions = [item.caption for item in image_items]

if captions:
    tfidf_vectorizer = TfidfVectorizer()
    tfidf_matrix = tfidf_vectorizer.fit_transform(captions)
else:
    tfidf_vectorizer = None
    tfidf_matrix = None

def tfidf_retrieve(query: str, top_k: int = 3) -> List[Tuple[int, float]]:
    if tfidf_vectorizer is None or tfidf_matrix is None or len(image_items) == 0:
        return []
    q_vec = tfidf_vectorizer.transform([query])
    scores = (tfidf_matrix @ q_vec.T).toarray().flatten()
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(i), float(scores[i])) for i in top_indices if scores[i] > 0]

print("BM25 index size:", len(tokenized_corpus))
print("FAISS index size:", faiss_index.ntotal)
print("TF-IDF caption index size:", len(captions))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 984.55it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 2/2 [00:01<00:00,  1.58it/s]

BM25 index size: 64
FAISS index size: 64
TF-IDF caption index size: 0


In [28]:
import os
import time
import pandas as pd
from google import genai
from google.genai import types
from dotenv import load_dotenv

# Load your secure API key
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError("GEMINI_API_KEY is missing! Check your .env file.")

# Configure the NEW Gemini client
client = genai.Client(api_key=api_key)

# --- Tuning knobs ---
TOP_K_RETRIEVAL = 20
TOP_K_EVIDENCE  = 15   

QUERY_EXPANSIONS = {
    "Q1": "advertising fee percentage average transaction gross revenue monthly",
    "Q2": "termination cause breach cure written notice days",
    "Q3": "putter equipment exception US Open other manufacturer brand exclusivity",
}

def build_context(text_hits, image_hits, max_chars: int = 150000) -> str:
    parts = []
    total = 0
    for idx, score in text_hits:
        chunk = page_chunks[idx]
        snippet = chunk.text  # FIX: Send the ENTIRE chunk!
        entry = f"[TEXT | {chunk.chunk_id}] {snippet}"
        if total + len(entry) > max_chars:
            break
        parts.append(entry)
        total += len(entry)
    for idx, score in image_hits:
        item = image_items[idx]
        entry = f"[IMAGE | {item.item_id}] Caption: {item.caption}"
        if total + len(entry) > max_chars:
            break
        parts.append(entry)
        total += len(entry)
    return "\n\n".join(parts)

def answer_with_gemini(question: str, context: str) -> str:
    prompt = f"""You are a precise legal document analyst. 
Answer the question using ONLY the provided context. 
If the answer is not in the context, say 'Not found in context.'

Context:
{context}

Question: {question}

Answer:"""
    
    try:
        # Using the new SDK syntax and the 2.5-flash model
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(
                temperature=0.0,
                max_output_tokens=1024
            )
        )
        return response.text.strip()
    except Exception as e:
        print(f"Gemini call failed: {e}")
        return "ERROR: Gemini call failed."

def check_keywords(text: str, keywords) -> list:
    text_lower = text.lower()
    return [kw for kw in keywords if kw.lower() in text_lower]

results = []

for query in QUERIES:
    qid      = query["id"]
    question = query["question"]
    must_kws = query["rubric"]["must_have_keywords"]
    opt_kws  = query["rubric"].get("optional_keywords", [])

    expanded_query = question + " " + QUERY_EXPANSIONS.get(qid, "")

    text_hits  = hybrid_retrieve(expanded_query, top_k=TOP_K_RETRIEVAL, alpha=0.5)
    image_hits = tfidf_retrieve(expanded_query, top_k=3)

    # Precision@5
    top5_hits = text_hits[:5]
    p5_hits = sum(
        1 for idx, _ in top5_hits
        if check_keywords(page_chunks[idx].text, must_kws)
    )
    precision_at_5 = p5_hits / 5

    # Recall@10
    all_top10_text = " ".join(page_chunks[idx].text for idx, _ in text_hits[:10])
    found_kws    = check_keywords(all_top10_text, must_kws)
    recall_at_10 = len(found_kws) / len(must_kws) if must_kws else 0.0

    context = build_context(text_hits[:TOP_K_EVIDENCE], image_hits)
    
    # Call Gemini 2.5 Flash
    answer  = answer_with_gemini(question, context)

    opt_hits_in_answer = check_keywords(answer, opt_kws)

    results.append({
        "query_id":          qid,
        "question":          question,
        "precision_at_5":    round(precision_at_5, 3),
        "recall_at_10":      round(recall_at_10, 3),
        "must_kws_found":    found_kws,
        "opt_kws_in_answer": opt_hits_in_answer,
        "gemini_answer":     answer,
        "num_text_hits":     len(text_hits),
        "num_image_hits":    len(image_hits)
    })

    print(f"\n{'='*60}")
    print(f"[{qid}] {question}")
    print(f"  Precision@5 : {precision_at_5:.3f}")
    print(f"  Recall@10   : {recall_at_10:.3f}")
    print(f"  Must-KWs    : {found_kws}")
    print(f"  Answer      : {answer}")
    
    time.sleep(2)

eval_df = pd.DataFrame(results)
eval_df.to_csv("phase3_eval_results_gemini.csv", index=False)
print("\n\nEvaluation Results:")
print(eval_df[["query_id", "precision_at_5", "recall_at_10", "must_kws_found"]].to_string(index=False))


[Q1] Under the Co-Branding Agreement, what is the exact formula used to calculate the advertising fees i-Escrow must pay 2TheMart?
  Precision@5 : 0.200
  Recall@10   : 1.000
  Must-KWs    : ['0.025%', 'average transaction']
  Answer      : After the Launch Date, i-Escrow shall pay 2TheMart advertising fees based on the number of Transaction Inquiries. This advertising fees shall consist of a per Transaction Inquiry amount calculated by multiplying 0.025% by the amount of the average Transaction from all Customers in the preceding quarter.

[Q2] What are the specific conditions and timeline under which Adams Golf can terminate the endorsement agreement with Tom Watson for cause?
  Precision@5 : 0.600
  Recall@10   : 0.667
  Must-KWs    : ['material breach', 'written notice']
  Answer      : Adams Golf can terminate the endorsement agreement with Tom Watson for cause under the following specific conditions and timelines:

1.  **Material Breach (Paragraph 23):**
    *   **

[Q3] Accordi